
## 1. Teste do cliente HTTP

### Objetivo
Validar o funcionamento do componente `APIClient`, responsável pelas requisições HTTP do pipeline.

### Implementação
O teste utilizará o endpoint de localizações de monitoramento da USGS, solicitando apenas um registro.

### Validações
1. Importação do módulo.
2. Execução da requisição HTTP.
3. Verificação do status da resposta.
4. Identificação da estrutura JSON retornada.

### Resultado esperado
Confirmar que o cliente HTTP consegue consumir uma API pública e retornar os dados em formato JSON.


In [1]:

import sys
from pathlib import Path

# Adiciona a raiz do projeto ao caminho de importação
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Importa o cliente HTTP
from src.extraction.api_client import APIClient

# Endpoint de localizações USGS
USGS_URL = (
    "https://api.waterdata.usgs.gov/ogcapi/v1/"
    "collections/monitoring-locations/items"
)

# Instancia o cliente
client = APIClient()

# Executa a consulta
dados_teste = client.get_json(
    USGS_URL,
    params={"limit": 1}
)

# Valida o resultado
print("Tipo de retorno:", dados_teste.get("type"))
print("Registros retornados:", dados_teste.get("numberReturned"))
print("Cliente HTTP:", type(client).__name__)


Tipo de retorno: FeatureCollection
Registros retornados: 1
Cliente HTTP: APIClient



### Conclusão: Teste do cliente HTTP

O módulo `APIClient` foi importado e executado com sucesso. A requisição ao endpoint de localizações da USGS retornou HTTP 200 e um registro no formato FeatureCollection.

O teste confirmou o funcionamento da comunicação HTTP e da conversão da resposta JSON, validando o componente que será reutilizado pelos módulos de extração do pipeline.

**Resultado:** cliente HTTP validado para continuidade da implementação.


In [2]:

from typing import Any

import pandas as pd

from src.extraction.api_client import APIClient


USGS_URL = (
    "https://api.waterdata.usgs.gov/ogcapi/v1/"
    "collections/monitoring-locations/items"
)


def get_monitoring_locations(
    client: APIClient,
    limit: int = 100,
) -> pd.DataFrame:
    """
    Consulta localizações de monitoramento disponibilizadas pela USGS.

    Args:
        client: Cliente HTTP utilizado na requisição.
        limit: Quantidade máxima de registros solicitados.

    Returns:
        DataFrame com propriedades e coordenadas das localizações.
    """
    response: dict[str, Any] = client.get_json(
        USGS_URL,
        params={"limit": limit},
    )

    features = response.get("features", [])

    if not features:
        return pd.DataFrame()

    df = pd.json_normalize(features)

    # Extração das coordenadas GeoJSON: longitude e latitude
    coordinates = df["geometry.coordinates"]

    df["longitude"] = coordinates.apply(
        lambda value: value[0] if isinstance(value, list) else None
    )

    df["latitude"] = coordinates.apply(
        lambda value: value[1] if isinstance(value, list) else None
    )

    return df



## 2. Extração de localizações USGS

### Objetivo
Implementar e testar a extração das localizações de monitoramento da USGS utilizando o cliente HTTP desenvolvido anteriormente.

### Resultado esperado
Obter um DataFrame com as propriedades das localizações e suas coordenadas geográficas.


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.extraction.api_client import APIClient
from src.extraction.usgs_client import get_monitoring_locations

client = APIClient()

df_usgs = get_monitoring_locations(client, limit=100)

print("Total de registros:", len(df_usgs))
print(
    "Coordenadas ausentes:",
    df_usgs[["longitude", "latitude"]].isna().any(axis=1).sum()
)

display(df_usgs.head(10))

Total de registros: 100
Coordenadas ausentes: 100


,type,id,geometry,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,...,properties.national_aquifer_code,properties.aquifer_type_code,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified,longitude,latitude
0,Feature,AL012-90100100001,None,AL012-90100100001,AL012,Alabama Office of Water Resources,90100100001,Autauga County Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None
1,Feature,AL012-90100100002,None,AL012-90100100002,AL012,Alabama Office of Water Resources,90100100002,Autaugaville Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
2,Feature,AL012-90100100003,None,AL012-90100100003,AL012,Alabama Office of Water Resources,90100100003,Billingsley Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
3,Feature,AL012-90100100005,None,AL012-90100100005,AL012,Alabama Office of Water Resources,90100100005,Water Works Board Of Prattville,01,US,...,None,None,None,None,None,None,None,None,None,None
4,Feature,AL012-90100100006,None,AL012-90100100006,AL012,Alabama Office of Water Resources,90100100006,West Autauga Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None
5,Feature,AL012-90100300001,None,AL012-90100300001,AL012,Alabama Office of Water Resources,90100300001,North Baldwin Utilities,01,US,...,None,None,None,None,None,None,None,None,None,None
6,Feature,AL012-90100300002,None,AL012-90100300002,AL012,Alabama Office of Water Resources,90100300002,Belforest Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
7,Feature,AL012-90100300004,None,AL012-90100300004,AL012,Alabama Office of Water Resources,90100300004,Daphne Utilities Board,01,US,...,None,None,None,None,None,None,None,None,None,None
8,Feature,AL012-90100300005,None,AL012-90100300005,AL012,Alabama Office of Water Resources,90100300005,Elberta Water Works,01,US,...,None,None,None,None,None,None,None,None,None,None
9,Feature,AL012-90100300006,None,AL012-90100300006,AL012,Alabama Office of Water Resources,90100300006,Fairhope Water Department,01,US,...,None,None,None,None,None,None,None,None,None,None



### Conclusão: Extração de localizações USGS

A extração foi executada com sucesso, retornando 100 registros e 48 colunas.

Durante a validação, identificamos que os registros consultados não apresentaram geometria geográfica (`geometry = None`).

Os dados cadastrais foram preservados. A disponibilidade de coordenadas será considerada na seleção das localizações utilizadas na integração meteorológica.

**Resultado:** módulo de extração USGS implementado e testado com sucesso.


Etapa 3: Implementação do módulo de extração Open-Meteo
1. Explicação
Vamos criar o módulo open_meteo_client.py para consultar dados meteorológicos horários.

Ele reutilizará o APIClient e receberá latitude e longitude como parâmetros. Assim, poderemos consultar diferentes localizações sem duplicar código.
2. Criar o arquivo
Localização: src/extraction/open_meteo_client.py

In [4]:

import pandas as pd

from src.extraction.api_client import APIClient


OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"


def get_weather_data(
    client: APIClient,
    latitude: float,
    longitude: float,
) -> pd.DataFrame:
    """Consulta dados meteorológicos horários da Open-Meteo."""

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
        ],
        "forecast_days": 1,
        "timezone": "auto",
    }

    response = client.get_json(
        OPEN_METEO_URL,
        params=params,
    )

    hourly_data = response.get("hourly", {})

    if not hourly_data:
        return pd.DataFrame()

    return pd.DataFrame(hourly_data)



## 3. Extração de dados meteorológicos Open-Meteo

### Objetivo
Implementar e testar a consulta de dados meteorológicos utilizando a API Open-Meteo.

### Justificativa
A Open-Meteo disponibiliza dados meteorológicos por coordenadas geográficas. Nesta etapa, utilizaremos o cliente HTTP desenvolvido anteriormente para consultar variáveis meteorológicas de uma localização selecionada.

### Estratégia
1. Importar a função de extração Open-Meteo.
2. Definir latitude e longitude da consulta.
3. Executar a requisição HTTP.
4. Converter os dados horários em DataFrame.
5. Verificar os registros e as variáveis retornadas.

### Resultado esperado
Obter dados meteorológicos horários organizados em DataFrame, contendo temperatura, umidade relativa, precipitação e velocidade do vento.


In [5]:
from src.extraction.open_meteo_client import get_weather_data

# Coordenadas de uma localização USGS com geometria disponível
latitude = 33.677862
longitude = -106.995028

# Consulta meteorológica
df_meteo = get_weather_data(
    client,
    latitude,
    longitude,
)

# Resumo da extração
print("Total de registros:", len(df_meteo))
print("Variáveis disponíveis:", df_meteo.columns.tolist())

display(df_meteo.head(10))


Total de registros: 24
Variáveis disponíveis: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-11-19T00:00,5.9,72,0.0,2.8
1,2026-11-19T01:00,5.3,72,0.0,2.3
2,2026-11-19T02:00,4.7,70,0.0,2.9
3,2026-11-19T03:00,4.0,76,0.0,4.6
4,2026-11-19T04:00,3.7,80,0.0,6.3
5,2026-11-19T05:00,3.3,81,0.0,6.2
6,2026-11-19T06:00,2.6,84,0.0,3.7
7,2026-11-19T07:00,2.3,81,0.0,3.5
8,2026-11-19T08:00,4.9,85,0.0,6.1
9,2026-11-19T09:00,8.0,72,0.0,8.2



### Conclusão: Extração Open-Meteo

O módulo `open_meteo_client.py` foi implementado para consultar dados meteorológicos horários por meio da API Open-Meteo.

A resposta foi convertida em DataFrame, organizando as variáveis meteorológicas selecionadas para utilização nas próximas etapas do pipeline.

**Resultado:** módulo de extração meteorológica implementado e validado.



## 4. Seleção de localizações com coordenadas geográficas

### Objetivo
Identificar localizações USGS que possuam coordenadas geográficas válidas para permitir a consulta meteorológica.

### Estratégia
Ampliar a extração para 1.000 registros e filtrar as localizações que possuem latitude e longitude.

### Resultado esperado
Obter um conjunto de localizações georreferenciadas para utilização na integração com a Open-Meteo.


In [6]:

# Consulta ampliada à USGS
df_usgs = get_monitoring_locations(client, limit=1000)

# Seleciona somente registros com coordenadas
df_usgs_validos = df_usgs.dropna(
    subset=["latitude", "longitude"]
).copy()

print("Localizações consultadas:", len(df_usgs))
print("Localizações com coordenadas:", len(df_usgs_validos))
print("Coordenadas ausentes:", len(df_usgs) - len(df_usgs_validos))

display(
    df_usgs_validos[
        [
            "properties.monitoring_location_number",
            "properties.monitoring_location_name",
            "properties.state_name",
            "longitude",
            "latitude",
        ]
    ].head(10)
)


Localizações consultadas: 1000
Localizações com coordenadas: 694
Coordenadas ausentes: 306


,properties.monitoring_location_number,properties.monitoring_location_name,properties.state_name,longitude,latitude
306,330110092432102,19S16W35DD1,Arkansas,-92.722658,33.019583
307,330335091341402,19S05W12DC1,Arkansas,-91.570676,33.059847
308,330410092015901,19S09W09DB1,Arkansas,-92.033188,33.069569
309,330545093113001,6058emerson ww,Arkansas,-93.191833,33.095968
310,330545093115001,6058 EMERSON,Arkansas,-93.197389,33.095968
311,330643092403701,18S16W31CB1,Arkansas,-92.677101,33.112079
312,330643092403702,18S16W32BC1,Arkansas,-92.677101,33.112079
313,330643092403703,16W18SBC3,Arkansas,-92.677101,33.112079
314,330645093274001,19S23W11BA1,Arkansas,-93.461284,33.112633
315,331000092562701,18S08W04CC2,Arkansas,-91.940965,33.166788



### Conclusão: Seleção de localizações georreferenciadas

A consulta ampliada à API USGS retornou 1.000 localizações, das quais 694 possuem coordenadas geográficas válidas.

Os registros sem coordenadas foram separados, mantendo apenas as localizações aptas à consulta meteorológica.

**Resultado:** conjunto de 694 localizações georreferenciadas disponível para a integração com a Open-Meteo.



## 5. Integração dos dados USGS e Open-Meteo

### Objetivo
Relacionar as informações cadastrais das localizações USGS aos dados meteorológicos obtidos pela Open-Meteo.

### Estratégia
1. Selecionar cinco localizações com coordenadas válidas.
2. Consultar os dados meteorológicos de cada localização.
3. Associar os registros meteorológicos aos identificadores das estações USGS.
4. Consolidar os resultados em um único DataFrame.

### Resultado esperado
Obter uma base integrada contendo identificação da localização, informações geográficas e variáveis meteorológicas.


In [7]:

# Seleção de cinco localizações para teste
amostra_usgs = df_usgs_validos.head(5).copy()

resultados = []

# Consulta meteorológica por localização
for _, local in amostra_usgs.iterrows():

    df_clima = get_weather_data(
        client,
        latitude=local["latitude"],
        longitude=local["longitude"],
    )

    if not df_clima.empty:
        df_clima["codigo_localizacao"] = (
            local["properties.monitoring_location_number"]
        )
        df_clima["nome_localizacao"] = (
            local["properties.monitoring_location_name"]
        )
        df_clima["estado"] = local["properties.state_name"]
        df_clima["latitude"] = local["latitude"]
        df_clima["longitude"] = local["longitude"]

        resultados.append(df_clima)

# Consolidação dos resultados
df_integrado = pd.concat(resultados, ignore_index=True)

print("Localizações processadas:", len(resultados))
print("Total de registros integrados:", len(df_integrado))
print("Total de colunas:", len(df_integrado.columns))

display(df_integrado.head(10))


Localizações processadas: 5
Total de registros integrados: 120
Total de colunas: 10


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
0,2026-11-19T00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
1,2026-11-19T01:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19T02:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19T03:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19T04:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
5,2026-11-19T05:00,9.9,79,0.0,1.9,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
6,2026-11-19T06:00,9.3,77,0.0,5.6,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
7,2026-11-19T07:00,8.7,78,0.0,6.6,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
8,2026-11-19T08:00,10.8,78,0.0,2.6,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
9,2026-11-19T09:00,15.9,54,0.0,4.8,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Integração USGS e Open-Meteo

A integração entre as APIs USGS e Open-Meteo foi executada com sucesso.

Foram processadas cinco localizações georreferenciadas, resultando em 120 registros meteorológicos distribuídos em 24 horários por localização.

Os dados foram consolidados em um único DataFrame, contendo informações cadastrais, geográficas e meteorológicas.

**Resultado:** primeira base integrada das fontes USGS e Open-Meteo disponível para as próximas etapas do pipeline.



## 6. Validação da qualidade dos dados integrados

### Objetivo
Verificar a consistência e a completude dos dados resultantes da integração entre as APIs USGS e Open-Meteo.

### Justificativa
Antes de armazenar os dados, precisamos identificar possíveis valores ausentes, registros duplicados e diferenças na quantidade de registros por localização.

### Estratégia
1. Verificar valores ausentes nas colunas.
2. Identificar duplicidades considerando a localização e o horário.
3. Contabilizar os registros por localização.
4. Conferir a quantidade total de registros e localizações processadas.

### Resultado esperado
Confirmar que a base integrada possui registros consistentes, sem duplicidades e com a quantidade esperada de observações por localização.


In [8]:
# Verificação de valores ausentes
print("Valores ausentes por coluna:")
display(df_integrado.isna().sum())

# Verificação de duplicidades
duplicados = df_integrado.duplicated(
    subset=["codigo_localizacao", "time"]
).sum()

print("\nRegistros duplicados:", duplicados)

# Quantidade de registros por localização
registros_por_local = (
    df_integrado.groupby("codigo_localizacao")
    .size()
    .reset_index(name="quantidade_registros")
)

display(registros_por_local)

# Resumo da validação
print("Total de registros:", len(df_integrado))
print(
    "Total de localizações:",
    df_integrado["codigo_localizacao"].nunique()
)
print(
    "Localizações com 24 registros:",
    (registros_por_local["quantidade_registros"] == 24).sum()
)


Valores ausentes por coluna:


time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
codigo_localizacao      0
nome_localizacao        0
estado                  0
latitude                0
longitude               0
dtype: int64


Registros duplicados: 0


,codigo_localizacao,quantidade_registros
0,330110092432102,24
1,330335091341402,24
2,330410092015901,24
3,330545093113001,24
4,330545093115001,24


Total de registros: 120
Total de localizações: 5
Localizações com 24 registros: 5



### Conclusão: Validação da qualidade dos dados

A base integrada foi submetida a verificações de qualidade, considerando valores ausentes, registros duplicados e quantidade de observações por localização.

Os testes identificaram 120 registros distribuídos entre cinco localizações, cada uma com 24 observações horárias.

Não foram encontrados valores ausentes ou registros duplicados.

**Resultado:** base integrada validada e pronta para a etapa de armazenamento.



## 7. Armazenamento dos dados integrados

### Objetivo
Persistir a base integrada em arquivo CSV para permitir sua reutilização em outras etapas do projeto.

### Justificativa
O armazenamento dos dados evita a necessidade de executar novamente todas as consultas às APIs sempre que precisarmos utilizar a base.

### Estratégia
1. Definir o diretório de saída.
2. Criar a pasta caso ainda não exista.
3. Exportar o DataFrame integrado para CSV.
4. Verificar a existência e o tamanho do arquivo gerado.

### Resultado esperado
Disponibilizar um arquivo CSV com os dados USGS e Open-Meteo integrados e validados.


In [9]:

from pathlib import Path

# Define o diretório e o arquivo de saída
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARQUIVO_SAIDA = OUTPUT_DIR / "usgs_open_meteo_integrado.csv"

# Exporta a base integrada
df_integrado.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig"
)

# Confirma o armazenamento
print("Arquivo gerado:", ARQUIVO_SAIDA)
print("Arquivo existe:", ARQUIVO_SAIDA.exists())
print("Tamanho:", round(ARQUIVO_SAIDA.stat().st_size / 1024, 2), "KB")


Arquivo gerado: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\usgs_open_meteo_integrado.csv
Arquivo existe: True
Tamanho: 12.61 KB



### Conclusão: Armazenamento dos dados integrados

A base integrada entre USGS e Open-Meteo foi exportada com sucesso para o formato CSV.

O arquivo `usgs_open_meteo_integrado.csv` foi armazenado no diretório `output/`, com tamanho aproximado de 12,62 KB.

**Resultado:** dados integrados e validados persistidos em arquivo, disponíveis para reutilização nas próximas etapas.



## 8. Verificação da base armazenada

### Objetivo
Confirmar que os dados exportados para CSV podem ser recuperados corretamente.

### Justificativa
A gravação bem-sucedida não garante, por si só, que o conteúdo armazenado possa ser recuperado sem alterações. Vamos realizar a leitura do arquivo e comparar sua estrutura com a base original.

### Estratégia
1. Ler o arquivo CSV utilizando Pandas.
2. Conferir a quantidade de registros e colunas.
3. Verificar os identificadores das localizações.
4. Comparar a estrutura recuperada com a base integrada.

### Resultado esperado
Recuperar os 120 registros e as 10 colunas da base integrada, confirmando a persistência dos dados.


In [10]:

# Leitura do arquivo armazenado
df_verificacao = pd.read_csv(ARQUIVO_SAIDA)

# Conferência da estrutura
print("Registros recuperados:", len(df_verificacao))
print("Colunas recuperadas:", len(df_verificacao.columns))
print(
    "Localizações recuperadas:",
    df_verificacao["codigo_localizacao"].nunique()
)

# Comparação com a base original
print(
    "Quantidade de registros preservada:",
    len(df_verificacao) == len(df_integrado)
)

print(
    "Quantidade de colunas preservada:",
    len(df_verificacao.columns) == len(df_integrado.columns)
)

display(df_verificacao.head())


Registros recuperados: 120
Colunas recuperadas: 10
Localizações recuperadas: 5
Quantidade de registros preservada: True
Quantidade de colunas preservada: True


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
0,2026-11-19T00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
1,2026-11-19T01:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19T02:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19T03:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19T04:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Verificação da base armazenada

A leitura do arquivo CSV foi realizada com sucesso utilizando Pandas.

Foram recuperados os 120 registros, as 10 colunas e as cinco localizações presentes na base original.

As verificações confirmaram a preservação da quantidade de registros e da estrutura dos dados após a exportação.

**Resultado:** armazenamento e recuperação da base integrada validados.



## 9. Encapsulamento do pipeline de dados

### Objetivo
Organizar as operações de extração, integração e validação em uma função centralizada.

### Justificativa
O encapsulamento permite executar o processo de forma padronizada, reduzindo a dependência de comandos isolados no notebook e facilitando sua manutenção.

### Estratégia
1. Criar uma função para executar o pipeline.
2. Reutilizar os módulos de extração USGS e Open-Meteo.
3. Consolidar os dados meteorológicos com as informações das localizações.
4. Retornar a base integrada para armazenamento.

### Resultado esperado
Disponibilizar uma função capaz de executar o fluxo de extração e integração de forma organizada e reutilizável.


In [11]:

import pandas as pd

from src.extraction.api_client import APIClient
from src.extraction.usgs_client import get_monitoring_locations
from src.extraction.open_meteo_client import get_weather_data


def run_pipeline(
    limit: int = 1000,
    max_locations: int = 5,
) -> pd.DataFrame:
    """Executa a extração e integração dos dados USGS e Open-Meteo."""

    client = APIClient()

    # Extração das localizações USGS
    df_usgs = get_monitoring_locations(client, limit=limit)

    # Seleção de localizações com coordenadas
    df_validos = df_usgs.dropna(
        subset=["latitude", "longitude"]
    ).head(max_locations)

    resultados = []

    # Extração meteorológica e integração
    for _, local in df_validos.iterrows():

        df_clima = get_weather_data(
            client,
            latitude=local["latitude"],
            longitude=local["longitude"],
        )

        if df_clima.empty:
            continue

        df_clima["codigo_localizacao"] = (
            local["properties.monitoring_location_number"]
        )
        df_clima["nome_localizacao"] = (
            local["properties.monitoring_location_name"]
        )
        df_clima["estado"] = local["properties.state_name"]
        df_clima["latitude"] = local["latitude"]
        df_clima["longitude"] = local["longitude"]

        resultados.append(df_clima)

    if not resultados:
        return pd.DataFrame()

    return pd.concat(resultados, ignore_index=True)


In [12]:
import sys
from pathlib import Path

# Configuração do caminho do projeto
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importação do pipeline
from src.extraction.pipeline import run_pipeline

# Execução
df_pipeline = run_pipeline(
    limit=1000,
    max_locations=5,
)

print("Total de registros:", len(df_pipeline))
print("Localizações:", df_pipeline["codigo_localizacao"].nunique())

display(df_pipeline.head())


Total de registros: 120
Localizações: 5


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
0,2026-11-19T00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
1,2026-11-19T01:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19T02:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19T03:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19T04:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Encapsulamento do pipeline

O fluxo de extração e integração foi organizado na função `run_pipeline()`, localizada no módulo `src/extraction/pipeline.py`.

O teste confirmou a execução das consultas às APIs USGS e Open-Meteo, com processamento de cinco localizações e geração de 120 registros integrados.

**Resultado:** pipeline encapsulado e validado, permitindo executar o processo por meio de uma única função.



## 10. Execução automatizada do pipeline

### Objetivo
Disponibilizar uma rotina de execução do pipeline com armazenamento automático dos dados integrados.

### Justificativa
A execução por um arquivo Python independente facilita a reprodução do processo, reduz a dependência do Jupyter Notebook e aproxima o projeto de uma aplicação de Engenharia de Dados.

### Estratégia
1. Executar a função `run_pipeline()`.
2. Receber a base integrada.
3. Criar o diretório de saída, caso necessário.
4. Exportar os dados para CSV.
5. Exibir um resumo da execução.

### Resultado esperado
Executar o pipeline por um único comando e gerar automaticamente o arquivo CSV atualizado.



### Conclusão: Execução automatizada do pipeline

O pipeline foi executado com sucesso por meio do arquivo `run.py`, utilizando o PowerShell.

A execução processou cinco localizações, gerou 120 registros meteorológicos e armazenou automaticamente o resultado em `output/usgs_open_meteo_integrado.csv`.

**Resultado:** execução independente e armazenamento automatizado validados.



### Conclusão: Teste de reprodutibilidade

O pipeline foi executado novamente por meio do arquivo `run.py`, confirmando a capacidade de reproduzir o processo de extração, integração e armazenamento.

A segunda execução gerou 120 registros para cinco localizações e atualizou o arquivo CSV de saída sem erros.

**Resultado:** reprodutibilidade do pipeline validada.



## 12. Configuração do cliente HTTP

### Objetivo
Configurar um cliente HTTP reutilizável para realizar requisições às APIs públicas utilizadas no pipeline.

### Justificativa
A centralização das configurações HTTP permite padronizar as chamadas às APIs, controlar o tempo de espera das requisições e estabelecer parâmetros para novas tentativas em caso de falhas temporárias.

### Estratégia
1. Definir o tempo limite das requisições (Timeout).
2. Configurar a quantidade máxima de retentativas (Retry).
3. Estabelecer o intervalo mínimo entre requisições (Rate Limiting).
4. Inicializar uma sessão HTTP reutilizável.
5. Verificar as configurações implementadas.

### Resultado esperado
Disponibilizar um cliente HTTP centralizado e reutilizável para os módulos de extração do projeto.


In [13]:
from src.extraction.api_client import APIClient

client_teste = APIClient(
    timeout=30,
    max_retries=3,
    min_interval=1.0,
)

print("Timeout:", client_teste.timeout)
print("Tentativas adicionais:", client_teste.max_retries)
print("Intervalo mínimo:", client_teste.min_interval, "segundo(s)")


Timeout: 30
Tentativas adicionais: 3
Intervalo mínimo: 1.0 segundo(s)



### Conclusão: Configuração do cliente HTTP

O cliente HTTP foi configurado com timeout de 30 segundos, três tentativas adicionais e intervalo mínimo de um segundo entre requisições.

As configurações foram carregadas corretamente durante o teste.

**Resultado:** parâmetros de controle de requisições configurados. O comportamento de Retry e Rate Limiting será validado em testes específicos.



## 13. Tratamento de erros e retentativas (Retry)

### Objetivo
Implementar o tratamento adequado de erros nas requisições HTTP e retentativas automáticas para falhas temporárias na comunicação com as APIs.

### Justificativa
Falhas de conexão, indisponibilidade temporária dos serviços e limites de requisições podem interromper a execução do pipeline. O tratamento adequado desses cenários aumenta a confiabilidade do processo de extração.

### Estratégia
1. Identificar erros de conexão e timeout.
2. Implementar retentativas para erros temporários HTTP, como 429 e 5xx.
3. Aplicar espera progressiva (Backoff Exponencial) entre as tentativas.
4. Respeitar o cabeçalho `Retry-After`, quando disponibilizado pela API.
5. Interromper a execução imediatamente diante de erros HTTP permanentes, como 400 e 404.
6. Tratar respostas que não estejam em formato JSON válido.

### Resultado esperado
Disponibilizar um mecanismo de retentativas para falhas temporárias, evitando novas tentativas desnecessárias para erros permanentes.



### 13.1. Teste do cliente HTTP

### Objetivo
Verificar se o cliente HTTP atualizado pode ser importado e inicializado corretamente.

### Justificativa
Após implementar o tratamento de erros e as retentativas, precisamos confirmar que as configurações continuam disponíveis e que o módulo pode ser utilizado pelos componentes do pipeline.

### Estratégia
1. Importar a classe `APIClient`.
2. Criar uma instância do cliente.
3. Verificar o timeout configurado.
4. Conferir a quantidade máxima de retentativas.
5. Validar o intervalo mínimo entre requisições.

### Resultado esperado
Confirmar a inicialização do cliente HTTP e a disponibilidade das configurações implementadas.


In [14]:

from src.extraction.api_client import APIClient

client_teste = APIClient()

print("Cliente HTTP:", type(client_teste).__name__)
print("Timeout:", client_teste.timeout)
print("Tentativas adicionais:", client_teste.max_retries)
print("Intervalo mínimo:", client_teste.min_interval)


Cliente HTTP: APIClient
Timeout: 30
Tentativas adicionais: 3
Intervalo mínimo: 1.0



### Conclusão: Teste de configuração do cliente HTTP

O cliente HTTP atualizado foi importado e inicializado corretamente.

As configurações de timeout, quantidade de retentativas e intervalo mínimo entre requisições foram identificadas conforme os valores definidos na implementação.

**Resultado:** inicialização e configurações do cliente HTTP validadas. Os mecanismos de Retry e tratamento de erros ainda dependem de testes específicos para comprovação de seu comportamento.



## 14. Transformação de dados com DuckDB

### Objetivo
Preparar o ambiente para executar transformações de dados utilizando SQL por meio do DuckDB.

### Justificativa
O DuckDB é um banco de dados analítico que permite consultar e transformar dados utilizando SQL diretamente no Python, sem necessidade de um servidor externo.

### Estratégia
1. Instalar a biblioteca DuckDB no ambiente virtual do projeto.
2. Validar sua instalação e integração com Python.
3. Utilizar SQL para trabalhar com estruturas JSON e dados tabulares.

### Resultado esperado
Ambiente preparado para executar consultas SQL e realizar transformações de dados.



### 14.1. Teste de instalação do DuckDB

### Objetivo
Verificar se a biblioteca DuckDB foi instalada e pode ser utilizada pelo Python.

### Estratégia
1. Importar a biblioteca DuckDB.
2. Consultar sua versão instalada.
3. Criar uma conexão em memória.
4. Executar uma consulta SQL simples.

### Resultado esperado
Confirmar a integração entre Python e DuckDB por meio da execução de uma consulta SQL.


In [15]:
import duckdb

print("DuckDB instalado e importado com sucesso!")
print("Versão:", duckdb.__version__)

conexao = duckdb.connect(database=":memory:")

resultado = conexao.execute(
    "SELECT 'DuckDB funcionando!' AS mensagem"
).fetchdf()

print(resultado)

conexao.close()

DuckDB instalado e importado com sucesso!
Versão: 2.0.0
              mensagem
0  DuckDB funcionando!



### Conclusão: Preparação e validação do ambiente DuckDB

O DuckDB foi instalado e configurado no ambiente virtual do projeto FT03, com sua dependência registrada no arquivo `requirements.txt`.

A integração com Python foi validada por meio da importação da biblioteca, criação de uma conexão em memória e execução de uma consulta SQL. O resultado foi convertido corretamente para um DataFrame utilizando `fetchdf()`.

**Resultado:** instalação, configuração, integração com Python e execução de consultas SQL validadas. Ambiente preparado para iniciar as transformações de dados com DuckDB.



### 14.2. Exploração da estrutura JSON da USGS

### Objetivo
Identificar a estrutura JSON retornada pela API da USGS e localizar os atributos necessários para a transformação dos dados utilizando SQL.

### Justificativa
Antes de realizar o achatamento de estruturas JSON, precisamos compreender como os registros estão organizados e onde estão armazenadas suas propriedades e coordenadas geográficas.

### Estratégia
1. Consultar a API da USGS utilizando o cliente HTTP do projeto.
2. Identificar as chaves principais da resposta.
3. Localizar o array `features`.
4. Inspecionar um registro para identificar suas propriedades e estrutura geográfica.

### Resultado esperado
Conhecer a estrutura JSON original e identificar os campos que serão utilizados na transformação com DuckDB.


In [16]:
# Importa recursos para trabalhar com JSON e tipagem.
import json
from typing import Any

# Importa o cliente HTTP e o endpoint da USGS.
from src.extraction.api_client import APIClient
from src.extraction.usgs_client import USGS_URL

# Inicializa o cliente HTTP.
client = APIClient()

# Consulta a API da USGS solicitando cinco registros.
response: dict[str, Any] = client.get_json(
    USGS_URL,
    params={"limit": 5},
)

# Recupera o array de registros da resposta.
features: list[dict[str, Any]] = response.get("features", [])

# Apresenta as chaves principais da resposta.
print("Chaves da resposta:")
print(list(response.keys()))

# Apresenta a quantidade de registros retornados.
print("\nQuantidade de registros:", len(features))

# Verifica se existem registros para inspeção.
if features:

    # Recupera o primeiro registro do array.
    primeiro_registro = features[0]

    # Apresenta as chaves disponíveis no registro.
    print("\nChaves do primeiro registro:")
    print(list(primeiro_registro.keys()))

    # Apresenta as propriedades disponíveis.
    print("\nPropriedades do registro:")
    print(list(primeiro_registro.get("properties", {}).keys()))

    # Exibe o registro completo em formato JSON indentado.
    print("\nPrimeiro registro completo:")
    print(json.dumps(primeiro_registro, indent=4, ensure_ascii=False))

else:
    print("\nA API não retornou registros.")


Chaves da resposta:
['type', 'features', 'numberReturned', 'links', 'timeStamp']

Quantidade de registros: 5

Chaves do primeiro registro:
['type', 'properties', 'id', 'geometry']

Propriedades do registro:
['id', 'agency_code', 'agency_name', 'monitoring_location_number', 'monitoring_location_name', 'district_code', 'country_code', 'country_name', 'state_code', 'state_name', 'county_code', 'county_name', 'minor_civil_division_code', 'site_type_code', 'site_type', 'hydrologic_unit_code', 'basin_code', 'altitude', 'altitude_accuracy', 'altitude_method_code', 'altitude_method_name', 'vertical_datum', 'vertical_datum_name', 'horizontal_positional_accuracy_code', 'horizontal_positional_accuracy', 'horizontal_position_method_code', 'horizontal_position_method_name', 'original_horizontal_datum', 'original_horizontal_datum_name', 'drainage_area', 'contributing_drainage_area', 'time_zone_abbreviation', 'uses_daylight_savings', 'construction_date', 'aquifer_code', 'national_aquifer_code', 'aqui


### 14.3. Achatamento de estruturas JSON com DuckDB SQL

### Objetivo
Utilizar SQL para percorrer o array `features` da resposta da USGS e extrair os atributos dos registros em uma estrutura tabular.

### Justificativa
A resposta da API utiliza JSON com objetos e arrays aninhados. O DuckDB permite percorrer essas estruturas e extrair seus atributos utilizando funções SQL específicas para JSON.

### Estratégia
1. Converter a resposta da API em uma representação JSON.
2. Utilizar `json_each()` para percorrer o array `features`.
3. Utilizar `json_extract_string()` para acessar propriedades aninhadas.
4. Extrair longitude e latitude quando disponíveis.
5. Converter o resultado SQL em um DataFrame.

### Resultado esperado
Obter uma tabela com os atributos das localizações da USGS, demonstrando o achatamento da estrutura JSON por meio de SQL.


In [17]:

# Importa a biblioteca utilizada para trabalhar com JSON.
import json

# Importa o DuckDB para executar consultas SQL.
import duckdb

# Converte a resposta original da USGS para uma string JSON.
# A variável response foi criada na etapa anterior.
json_usgs = json.dumps(response)

# Cria uma conexão DuckDB em memória.
conexao = duckdb.connect(database=":memory:")

# Define a consulta SQL para percorrer e achatar o array features.
consulta_sql = """
    SELECT
        -- Identifica a posição do registro dentro do array.
        CAST(feature.key AS INTEGER) AS indice_registro,

        -- Extrai o identificador da localização.
        json_extract_string(
            feature.value,
            '$.properties.id'
        ) AS id,

        -- Extrai o código da agência responsável.
        json_extract_string(
            feature.value,
            '$.properties.agency_code'
        ) AS codigo_agencia,

        -- Extrai o nome da agência responsável.
        json_extract_string(
            feature.value,
            '$.properties.agency_name'
        ) AS nome_agencia,

        -- Extrai o número da localização de monitoramento.
        json_extract_string(
            feature.value,
            '$.properties.monitoring_location_number'
        ) AS codigo_localizacao,

        -- Extrai o nome da localização de monitoramento.
        json_extract_string(
            feature.value,
            '$.properties.monitoring_location_name'
        ) AS nome_localizacao,

        -- Extrai o nome do estado.
        json_extract_string(
            feature.value,
            '$.properties.state_name'
        ) AS estado,

        -- Extrai a longitude quando a geometria está disponível.
        -- TRY_CAST converte o valor para número ou retorna NULL.
        TRY_CAST(
            json_extract_string(
                feature.value,
                '$.geometry.coordinates[0]'
            ) AS DOUBLE
        ) AS longitude,

        -- Extrai a latitude quando a geometria está disponível.
        TRY_CAST(
            json_extract_string(
                feature.value,
                '$.geometry.coordinates[1]'
            ) AS DOUBLE
        ) AS latitude

    -- Percorre cada elemento do array features.
    FROM json_each(
        ?::JSON,
        '$.features'
    ) AS feature
"""

# Executa a consulta SQL passando o JSON como parâmetro.
# fetchdf() converte o resultado da consulta em DataFrame.
df_usgs_sql = conexao.execute(
    consulta_sql,
    [json_usgs],
).fetchdf()

# Apresenta o resultado da transformação.
print("Registros transformados:", len(df_usgs_sql))
display(df_usgs_sql)

# Encerra a conexão com o DuckDB.
conexao.close()


Registros transformados: 5


,indice_registro,id,codigo_agencia,nome_agencia,codigo_localizacao,nome_localizacao,estado,longitude,latitude
0,0,AL012-90100100001,AL012,Alabama Office of Water Resources,90100100001,Autauga County Water Authority,Alabama,NaN,NaN
1,1,AL012-90100100002,AL012,Alabama Office of Water Resources,90100100002,Autaugaville Water System,Alabama,NaN,NaN
2,2,AL012-90100100003,AL012,Alabama Office of Water Resources,90100100003,Billingsley Water System,Alabama,NaN,NaN
3,3,AL012-90100100005,AL012,Alabama Office of Water Resources,90100100005,Water Works Board Of Prattville,Alabama,NaN,NaN
4,4,AL012-90100100006,AL012,Alabama Office of Water Resources,90100100006,West Autauga Water Authority,Alabama,NaN,NaN



### 14.3.1. Teste do achatamento JSON com SQL

### Objetivo
Verificar se a consulta SQL conseguiu percorrer o array `features` e extrair corretamente os atributos da estrutura JSON.

### Estratégia
1. Conferir a quantidade de registros retornados.
2. Verificar as colunas extraídas.
3. Comparar os identificadores e nomes com a resposta original.
4. Verificar o tratamento de coordenadas ausentes.

### Resultado esperado
Obter cinco registros tabulares, com os atributos extraídos das propriedades JSON e coordenadas preenchidas quando disponíveis.



### Conclusão: Achatamento de estruturas JSON com DuckDB SQL

A consulta SQL utilizando DuckDB percorreu o array `features` da resposta da API USGS e extraiu os atributos dos objetos JSON aninhados.

Os dados foram organizados em uma estrutura tabular com cinco registros e nove colunas. A extração das propriedades, identificadores e nomes das localizações ocorreu corretamente.

As coordenadas geográficas foram retornadas como valores nulos, pois os registros utilizados na amostra não disponibilizavam informações de geometria.

**Resultado:** achatamento de estruturas JSON utilizando SQL com DuckDB validado.


## 15. Padronização de tipos e datas

### Objetivo
Criar um módulo responsável pela padronização dos tipos de dados do conjunto integrado.

### Justificativa
As informações provenientes das APIs precisam apresentar tipos consistentes para permitir operações de análise, validação, integração e persistência.

### Estratégia
1. Criar o módulo `transform.py`.
2. Implementar uma função para padronização dos tipos.
3. Converter datas para datetime.
4. Converter coordenadas e variáveis meteorológicas para tipos numéricos.
5. Preservar identificadores como valores textuais.
6. Validar os tipos resultantes.

### Resultado esperado
Disponibilizar uma função reutilizável para padronização dos dados no pipeline.



### 15.1. Teste da padronização de tipos e datas

### Objetivo
Verificar se a função `standardize_data_types()` converte corretamente as colunas do conjunto integrado.

### Estratégia
1. Carregar o CSV gerado pelo pipeline.
2. Importar a função de transformação.
3. Executar a padronização dos tipos.
4. Comparar os tipos antes e depois da transformação.
5. Validar as conversões de datas, identificadores e valores numéricos.

### Resultado esperado
Confirmar que as colunas possuem os tipos definidos na implementação e que o DataFrame original permanece preservado.


In [18]:

from pathlib import Path

# Identifica o diretório de trabalho atual do Jupyter.
diretorio_atual = Path.cwd()

print("Diretório atual:", diretorio_atual)
print("Pasta output existe:", (diretorio_atual.parent / "output").exists())


Diretório atual: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\notebooks
Pasta output existe: True


In [19]:
import pandas as pd
from pathlib import Path

from src.transformation.transform import standardize_data_types

# Localiza a raiz do projeto a partir da pasta notebooks.
PROJECT_ROOT = Path.cwd().parent

# Define o caminho completo do arquivo CSV.
OUTPUT_FILE = PROJECT_ROOT / "output" / "usgs_open_meteo_integrado.csv"

# Carrega os dados integrados gerados pelo pipeline.
df_original = pd.read_csv(OUTPUT_FILE)

# Exibe os tipos originais das colunas.
print("Tipos antes da transformação:")
print(df_original.dtypes)

# Executa a função de padronização.
df_transformado = standardize_data_types(df_original)

# Exibe os tipos após a transformação.
print("\nTipos após a transformação:")
print(df_transformado.dtypes)

# Valida a conversão da coluna de data e horário.
assert pd.api.types.is_datetime64_any_dtype(
    df_transformado["time"]
)

# Valida as coordenadas geográficas.
assert pd.api.types.is_numeric_dtype(
    df_transformado["latitude"]
)

assert pd.api.types.is_numeric_dtype(
    df_transformado["longitude"]
)

# Valida a identificação textual da localização.
assert isinstance(
    df_transformado["codigo_localizacao"].dtype,
    pd.StringDtype,
)

# Confirma que o DataFrame original não foi alterado.
assert not pd.api.types.is_datetime64_any_dtype(
    df_original["time"]
)

print("\nTodas as validações foram aprovadas.")

# Apresenta uma amostra dos dados transformados.
display(df_transformado.head())


Tipos antes da transformação:
time                        str
temperature_2m          float64
relative_humidity_2m      int64
precipitation           float64
wind_speed_10m          float64
codigo_localizacao        int64
nome_localizacao            str
estado                      str
latitude                float64
longitude               float64
dtype: object

Tipos após a transformação:
time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
wind_speed_10m                 float64
codigo_localizacao              string
nome_localizacao                string
estado                          string
latitude                       float64
longitude                      float64
dtype: object

Todas as validações foram aprovadas.


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
0,2026-11-19 00:00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
1,2026-11-19 01:00:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19 02:00:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19 03:00:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19 04:00:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Padronização de tipos e datas

A função `standardize_data_types()` foi implementada e validada com os dados integrados das APIs USGS e Open-Meteo.

A coluna de data e horário foi convertida para datetime, os identificadores foram padronizados como texto e as variáveis meteorológicas e coordenadas permaneceram em formatos numéricos adequados.

O teste também confirmou que o DataFrame original foi preservado durante a transformação.

**Resultado:** padronização de tipos e datas validada.



## 16. Tratamento de valores nulos

### Objetivo
Implementar uma função para identificar e tratar registros com valores ausentes no conjunto integrado.

### Justificativa
Valores nulos podem comprometer operações de integração, análise e validação. Entretanto, substituir indiscriminadamente valores ausentes pode distorcer informações meteorológicas.

### Estratégia
1. Identificar os campos essenciais para cada registro.
2. Remover registros sem data ou identificador da localização.
3. Preservar valores ausentes em campos descritivos e meteorológicos.
4. Comparar a quantidade de registros antes e depois do tratamento.

### Resultado esperado
Disponibilizar uma função reutilizável para tratar valores nulos sem atribuir valores artificiais às medições meteorológicas.



### 16.1. Teste do tratamento de valores nulos

### Objetivo
Verificar se a função `handle_null_values()` remove registros sem campos essenciais e preserva valores ausentes nas variáveis meteorológicas.

### Estratégia
1. Criar uma amostra de teste a partir dos dados integrados.
2. Simular a ausência de uma data e de uma medição meteorológica.
3. Executar a função de tratamento.
4. Verificar os registros removidos e os valores nulos preservados.
5. Confirmar que o DataFrame original não foi alterado.

### Resultado esperado
Remover registros sem identificação ou data, preservando valores meteorológicos ausentes e mantendo o conjunto original intacto.


In [20]:
import pandas as pd

from src.transformation.transform import (
    standardize_data_types,
    handle_null_values,
)

# Localiza a raiz do projeto.
PROJECT_ROOT = Path.cwd().parent

# Define o caminho do arquivo integrado.
OUTPUT_FILE = (
    PROJECT_ROOT
    / "output"
    / "usgs_open_meteo_integrado.csv"
)

# Carrega os dados originais.
df_original = pd.read_csv(OUTPUT_FILE)

# Padroniza os tipos antes do tratamento.
df_teste = standardize_data_types(df_original)

# Cria uma amostra independente para o teste.
df_teste = df_teste.copy()

# Simula a ausência de uma data em um registro.
df_teste.loc[0, "time"] = pd.NaT

# Simula a ausência de uma medição meteorológica.
df_teste.loc[1, "temperature_2m"] = None

# Executa o tratamento de valores nulos.
df_tratado = handle_null_values(df_teste)

# Verifica se o registro sem data foi removido.
assert df_tratado["time"].notna().all()

# Verifica se o valor meteorológico ausente foi preservado.
assert df_tratado["temperature_2m"].isna().any()

# Confirma que o DataFrame original não foi alterado.
assert len(df_original) == 120

# Confirma que somente um registro foi removido.
assert len(df_tratado) == len(df_teste) - 1

print("\nTodas as validações foram aprovadas.")

# Apresenta a quantidade de valores nulos após o tratamento.
print("\nValores nulos por coluna:")
print(df_tratado.isna().sum())

# Apresenta uma amostra do resultado.
display(df_tratado.head())


Registros antes do tratamento: 120
Registros removidos: 1
Registros após o tratamento: 119

Todas as validações foram aprovadas.

Valores nulos por coluna:
time                    0
temperature_2m          1
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
codigo_localizacao      0
nome_localizacao        0
estado                  0
latitude                0
longitude               0
dtype: int64


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
1,2026-11-19 01:00:00,NaN,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19 02:00:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19 03:00:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19 04:00:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
5,2026-11-19 05:00:00,9.9,79,0.0,1.9,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Tratamento de valores nulos

A função `handle_null_values()` foi implementada e validada com os dados integrados das APIs USGS e Open-Meteo.

O tratamento removeu registros sem informações essenciais de identificação ou data, preservando valores ausentes nas variáveis meteorológicas para evitar distorções nas medições.

No teste, um registro foi removido e um valor ausente de temperatura foi mantido. O DataFrame original permaneceu preservado.

**Resultado:** tratamento de valores nulos validado.



## 17. Remoção de registros duplicados

### Objetivo
Implementar uma função para identificar e remover registros duplicados no conjunto integrado.

### Justificativa
Registros duplicados podem distorcer contagens, cálculos estatísticos e análises temporais. A identificação de cada medição deve considerar sua localização e seu horário.

### Estratégia
1. Definir `codigo_localizacao` e `time` como chave de identificação.
2. Verificar a existência das colunas obrigatórias.
3. Identificar registros duplicados.
4. Preservar a primeira ocorrência.
5. Validar a unicidade dos registros após a transformação.

### Resultado esperado
Disponibilizar uma função reutilizável para remover duplicidades, mantendo apenas uma medição por localização e horário.



### 17.1. Teste da remoção de registros duplicados

### Objetivo
Verificar se a função `remove_duplicates()` identifica e remove corretamente registros repetidos.

### Estratégia
1. Carregar os dados integrados.
2. Padronizar os tipos de dados.
3. Aplicar o tratamento de valores nulos.
4. Criar uma duplicidade simulada.
5. Executar a função de remoção.
6. Verificar a unicidade da chave e a preservação do DataFrame original.

### Resultado esperado
Remover o registro duplicado simulado, preservando os registros válidos e garantindo a unicidade da combinação entre localização e horário.


In [21]:
import pandas as pd
from pathlib import Path

from src.transformation.transform import (
    standardize_data_types,
    handle_null_values,
    remove_duplicates,
)

# Identifica a raiz do projeto.
PROJECT_ROOT = Path.cwd().parent

# Define o caminho do arquivo CSV integrado.
OUTPUT_FILE = (
    PROJECT_ROOT
    / "output"
    / "usgs_open_meteo_integrado.csv"
)

# Carrega os dados originais.
df_original = pd.read_csv(OUTPUT_FILE)

# Padroniza os tipos de dados.
df_teste = standardize_data_types(df_original)

# Remove registros sem os campos essenciais.
df_teste = handle_null_values(df_teste)

# Cria uma cópia da primeira linha para simular uma duplicidade.
registro_duplicado = df_teste.iloc[[0]].copy()

# Adiciona o registro duplicado ao conjunto de teste.
df_teste = pd.concat(
    [df_teste, registro_duplicado],
    ignore_index=True,
)

# Registra a quantidade de linhas antes da remoção.
registros_antes = len(df_teste)

# Executa a remoção de duplicidades.
df_sem_duplicados = remove_duplicates(df_teste)

# Verifica se a chave de identificação é única.
assert not df_sem_duplicados.duplicated(
    subset=["codigo_localizacao", "time"]
).any()

# Confirma que exatamente um registro foi removido.
assert len(df_sem_duplicados) == registros_antes - 1

# Confirma que o DataFrame original continua com 120 registros.
assert len(df_original) == 120

print("\nTodas as validações foram aprovadas.")

# Apresenta uma amostra do resultado.
display(df_sem_duplicados.head())


Registros antes do tratamento: 120
Registros removidos: 0
Registros após o tratamento: 120
Registros antes da transformação: 121
Duplicidades identificadas: 1
Registros removidos: 1
Registros após a transformação: 120

Todas as validações foram aprovadas.


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude
0,2026-11-19 00:00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
1,2026-11-19 01:00:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
2,2026-11-19 02:00:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
3,2026-11-19 03:00:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658
4,2026-11-19 04:00:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658



### Conclusão: Remoção de registros duplicados

A função `remove_duplicates()` foi implementada e validada utilizando a combinação das colunas `codigo_localizacao` e `time` como chave de identificação dos registros.

O teste simulou a inclusão de um registro duplicado, que foi identificado e removido corretamente, preservando a primeira ocorrência e garantindo a unicidade da chave.

O DataFrame original permaneceu preservado durante a transformação.

**Resultado:** remoção de registros duplicados validada.



### 18. Teste da integração com DuckDB

**Objetivo:** verificar se o relacionamento entre os dados meteorológicos e os metadados das localizações está funcionando corretamente.

**Estratégia:** executar o pipeline de extração, separar os dados meteorológicos dos metadados das localizações e utilizar a função `join_data_sources()` para realizar o INNER JOIN.

**Resultado esperado:** obter 120 registros integrados, distribuídos entre as cinco localizações, sem perda de registros no relacionamento.


In [22]:

# Importa as funções necessárias para extração e integração.
from src.extraction.pipeline import run_pipeline
from src.transformation.transform import join_data_sources

# Executa a extração das fontes USGS e Open-Meteo.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Separa os dados meteorológicos.
df_weather = df_origem[
    [
        "time",
        "codigo_localizacao",
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
    ]
].copy()

# Separa os metadados das localizações.
df_locations = (
    df_origem[
        [
            "codigo_localizacao",
            "nome_localizacao",
            "estado",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates(subset=["codigo_localizacao"])
    .copy()
)

# Executa o relacionamento entre as fontes usando SQL INNER JOIN.
df_integrado = join_data_sources(
    df_weather=df_weather,
    df_locations=df_locations,
)

# Apresenta os resultados da integração.
print("Registros meteorológicos:", len(df_weather))
print("Localizações distintas:", df_locations["codigo_localizacao"].nunique())
print("Registros após o JOIN:", len(df_integrado))

# Exibe uma amostra dos dados integrados.
display(df_integrado.head())

# Verifica a distribuição dos registros por localização.
display(
    df_integrado.groupby("codigo_localizacao")
    .size()
    .reset_index(name="quantidade_registros")
)


Registros meteorológicos: 120
Localizações distintas: 5
Registros após o JOIN: 120


,time,codigo_localizacao,nome_localizacao,estado,latitude,longitude,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-11-19T00:00,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,11.4,84,0.0,5.2
1,2026-11-19T01:00,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,11.5,81,0.0,5.2
2,2026-11-19T02:00,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,11.4,81,0.0,0.4
3,2026-11-19T03:00,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,12.0,81,0.0,2.7
4,2026-11-19T04:00,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,10.6,82,0.0,1.4


,codigo_localizacao,quantidade_registros
0,330110092432102,24
1,330335091341402,24
2,330410092015901,24
3,330545093113001,24
4,330545093115001,24



**Conclusão:** a integração entre as fontes USGS e Open-Meteo foi realizada com sucesso utilizando SQL INNER JOIN no DuckDB. O relacionamento pela chave `codigo_localizacao` preservou os 120 registros meteorológicos, distribuídos entre cinco localizações, com 24 registros cada. O resultado contém as informações meteorológicas e os respectivos metadados das localizações.



## 19. Enriquecimento dos dados com métricas

**Objetivo:** enriquecer os dados meteorológicos com métricas derivadas das medições coletadas pela Open-Meteo.

**Implementação:** desenvolver funções em Python com Pandas para calcular métricas acumuladas e médias por localização, utilizando as variáveis de temperatura, precipitação e velocidade do vento.

**Justificativa:** demonstrar a aplicação de cálculos analíticos sobre os dados transformados, agregando informações que não são fornecidas diretamente na estrutura original da API.

**Resultado esperado:** gerar um DataFrame enriquecido com métricas adicionais, preservando os registros e permitindo análises por localização.



**Teste:** enriquecimento dos dados meteorológicos

O teste executa o pipeline de extração e integração, aplica a função de enriquecimento e verifica:

- A criação das três métricas adicionais.
- A quantidade de registros antes e depois do enriquecimento.
- A geração de métricas para as cinco localizações.
- A preservação dos dados originais.


In [23]:

# Importa a função responsável pelo enriquecimento dos dados.
from src.transformation.transform import enrich_with_metrics

# Utiliza o DataFrame integrado produzido no teste anterior.
# Caso o kernel tenha sido reiniciado, executa novamente o pipeline.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Aplica o enriquecimento das métricas por localização.
df_enriquecido = enrich_with_metrics(df_origem)

# Define as colunas das métricas calculadas.
metric_columns = [
    "temperatura_media_localizacao",
    "precipitacao_acumulada_localizacao",
    "velocidade_media_vento_localizacao",
]

# Apresenta a quantidade de registros antes e depois.
print("Registros originais:", len(df_origem))
print("Registros enriquecidos:", len(df_enriquecido))

# Verifica a quantidade de colunas.
print("Colunas originais:", len(df_origem.columns))
print("Colunas enriquecidas:", len(df_enriquecido.columns))

# Verifica se as três métricas foram criadas.
print("\nMétricas disponíveis:")
print(df_enriquecido[metric_columns].notna().sum())

# Exibe as métricas por localização.
display(
    df_enriquecido[
        ["codigo_localizacao"] + metric_columns
    ]
    .drop_duplicates(subset=["codigo_localizacao"])
    .reset_index(drop=True)
)

# Verifica se os registros originais foram preservados.
assert len(df_origem) == len(df_enriquecido)

# Verifica se as métricas foram adicionadas.
assert all(
    column in df_enriquecido.columns
    for column in metric_columns
)

# Confirma que o DataFrame original não foi alterado.
assert not any(
    column in df_origem.columns
    for column in metric_columns
)

print("\nTeste de enriquecimento concluído.")


Registros originais: 120
Registros enriquecidos: 120
Colunas originais: 10
Colunas enriquecidas: 13

Métricas disponíveis:
temperatura_media_localizacao         120
precipitacao_acumulada_localizacao    120
velocidade_media_vento_localizacao    120
dtype: int64


,codigo_localizacao,temperatura_media_localizacao,precipitacao_acumulada_localizacao,velocidade_media_vento_localizacao
0,330110092432102,15.675000,0.0,5.995833
1,330335091341402,16.429167,0.0,9.808333
2,330410092015901,16.041667,0.0,6.908333
3,330545093113001,16.404167,0.0,7.925000
4,330545093115001,16.112500,0.0,8.100000



Teste de enriquecimento concluído.



**Conclusão:** o enriquecimento dos dados meteorológicos foi realizado com sucesso utilizando Pandas. Foram calculadas três métricas por localização: temperatura média, precipitação acumulada e velocidade média do vento. O DataFrame passou de 10 para 13 colunas, preservando os 120 registros originais e as cinco localizações analisadas.



## 20. Validação da qualidade dos dados

**Objetivo:** avaliar a qualidade dos dados integrados e enriquecidos, verificando sua consistência e integridade.

**Estratégia:** implementar verificações utilizando Pandas para analisar a estrutura do DataFrame, a proporção de valores nulos, a unicidade dos registros e os intervalos dos valores numéricos.

**Resultado esperado:** obter um diagnóstico da qualidade dos dados, identificando possíveis inconsistências antes da persistência na camada final.



**Teste:** validação da qualidade dos dados

O teste executa o pipeline de extração, aplica o enriquecimento das métricas e avalia as quatro dimensões de qualidade implementadas:

- Estrutura: identificação de colunas ausentes.
- Completude: quantidade e proporção de valores nulos.
- Unicidade: identificação de registros duplicados.
- Validade: verificação dos intervalos numéricos definidos.

**Resultado esperado:** obter um relatório com os resultados das verificações de qualidade do conjunto de dados.


In [24]:

# Importa as funções necessárias para executar o teste.
from src.extraction.pipeline import run_pipeline
from src.transformation.transform import enrich_with_metrics
from src.quality.validate import validate_data_quality

# Executa a extração dos dados das fontes.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_origem)

# Executa as verificações de qualidade.
quality_report = validate_data_quality(df_enriquecido)

# Apresenta o resumo estrutural do conjunto de dados.
print("RESUMO DO CONJUNTO DE DADOS")
print("Total de registros:", quality_report["total_records"])
print("Total de colunas:", quality_report["total_columns"])
print("Colunas ausentes:", quality_report["structure"]["missing_columns"])
print("Status estrutural:", quality_report["structure"]["status"])

# Apresenta os resultados de completude.
print("\nCOMPLETUDE")
print("Total de valores nulos:", quality_report["completeness"]["total_null_values"])

df_nulos = pd.DataFrame({
    "coluna": quality_report["completeness"]["null_counts"].keys(),
    "valores_nulos": quality_report["completeness"]["null_counts"].values(),
    "percentual_nulos": quality_report["completeness"]["null_ratio_percent"].values(),
})

display(df_nulos)

# Apresenta o resultado de unicidade.
print("\nUNICIDADE")
print(
    "Registros duplicados:",
    quality_report["uniqueness"]["duplicate_records"]
)

# Apresenta os resultados de validade.
print("\nVALIDADE")

df_validade = pd.DataFrame.from_dict(
    quality_report["validity"],
    orient="index"
).reset_index()

df_validade.columns = [
    "variavel",
    "status",
    "valores_invalidos",
]

display(df_validade)

# Exibe o resultado geral das verificações.
print("\nValidação de qualidade concluída.")


RESUMO DO CONJUNTO DE DADOS
Total de registros: 120
Total de colunas: 13
Colunas ausentes: []
Status estrutural: válido

COMPLETUDE
Total de valores nulos: 0


,coluna,valores_nulos,percentual_nulos
0,time,0,0.0
1,temperature_2m,0,0.0
2,relative_humidity_2m,0,0.0
3,precipitation,0,0.0
4,wind_speed_10m,0,0.0
5,codigo_localizacao,0,0.0
6,nome_localizacao,0,0.0
7,estado,0,0.0
8,latitude,0,0.0
9,longitude,0,0.0



UNICIDADE
Registros duplicados: 0

VALIDADE


,variavel,status,valores_invalidos
0,latitude,válido,0
1,longitude,válido,0
2,temperature_2m,válido,0
3,relative_humidity_2m,válido,0
4,precipitation,válido,0
5,wind_speed_10m,válido,0



Validação de qualidade concluída.



**Conclusão:** a validação da qualidade dos dados foi concluída com sucesso. O conjunto de dados apresentou 120 registros e 13 colunas, sem colunas obrigatórias ausentes, valores nulos, registros duplicados ou valores fora dos intervalos definidos. As quatro dimensões de qualidade avaliadas apresentaram resultados satisfatórios.



## 21. Relatório de execução e auditoria de qualidade

**Objetivo:** registrar os resultados da execução do pipeline e das verificações de qualidade dos dados.

**Estratégia:** utilizar os resultados da extração, transformação e validação para gerar um relatório estruturado em formato JSON, contendo informações da execução, quantidade de registros, localização dos arquivos e resultados das verificações de qualidade.

**Resultado esperado:** gerar um arquivo JSON na pasta `output/`, permitindo consultar e auditar os resultados de cada execução do pipeline.



**Teste:** geração do relatório de execução e auditoria.

O teste executa o pipeline, aplica o enriquecimento das métricas, realiza a validação da qualidade e exporta os resultados em JSON.

Serão verificadas a existência do arquivo, sua estrutura e as informações registradas.


In [25]:

# Importa as bibliotecas e funções necessárias.
import json
from pathlib import Path

from src.extraction.pipeline import run_pipeline
from src.transformation.transform import enrich_with_metrics
from src.quality.validate import validate_data_quality
from src.quality.report import export_quality_report

# Define o diretório principal do projeto.
PROJECT_ROOT = Path.cwd().parent

# Define o diretório de saída dos arquivos.
OUTPUT_DIR = PROJECT_ROOT / "output"

# Executa a extração dos dados.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_origem)

# Executa a validação da qualidade dos dados.
quality_report = validate_data_quality(df_enriquecido)

# Exporta o relatório de qualidade em formato JSON.
report_path = export_quality_report(
    quality_report=quality_report,
    output_dir=OUTPUT_DIR,
)

# Verifica se o arquivo foi criado.
assert report_path.exists(), "Arquivo JSON não foi gerado."

# Abre o arquivo para conferir seu conteúdo.
with report_path.open("r", encoding="utf-8") as file:
    report_data = json.load(file)

# Apresenta as informações principais do relatório.
print("Relatório gerado com sucesso.")
print("Arquivo:", report_path)
print("Tamanho:", report_path.stat().st_size, "bytes")

print("\nRESUMO DA EXECUÇÃO")
print("Data e hora:", report_data["execution"]["timestamp"])
print("Registros:", report_data["execution"]["total_records"])
print("Colunas:", report_data["execution"]["total_columns"])

print("\nAUDITORIA DE QUALIDADE")
print(
    "Valores nulos:",
    report_data["data_quality"]["completeness"]["total_null_values"],
)
print(
    "Registros duplicados:",
    report_data["data_quality"]["uniqueness"]["duplicate_records"],
)
print(
    "Colunas ausentes:",
    report_data["data_quality"]["structure"]["missing_columns"],
)

# Exibe o conteúdo completo do relatório.
display(report_data)


Relatório gerado com sucesso.
Arquivo: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\quality_report.json
Tamanho: 2444 bytes

RESUMO DA EXECUÇÃO
Data e hora: 2026-11-19T21:30:49.206047+00:00
Registros: 120
Colunas: 13

AUDITORIA DE QUALIDADE
Valores nulos: 0
Registros duplicados: 0
Colunas ausentes: []


{'execution': {'timestamp': '2026-11-19T21:30:49.206047+00:00',
  'total_records': 120,
  'total_columns': 13},
 'data_quality': {'total_records': 120,
  'total_columns': 13,
  'structure': {'missing_columns': [], 'status': 'válido'},
  'completeness': {'null_counts': {'time': 0,
    'temperature_2m': 0,
    'relative_humidity_2m': 0,
    'precipitation': 0,
    'wind_speed_10m': 0,
    'codigo_localizacao': 0,
    'nome_localizacao': 0,
    'estado': 0,
    'latitude': 0,
    'longitude': 0,
    'temperatura_media_localizacao': 0,
    'precipitacao_acumulada_localizacao': 0,
    'velocidade_media_vento_localizacao': 0},
   'null_ratio_percent': {'time': 0.0,
    'temperature_2m': 0.0,
    'relative_humidity_2m': 0.0,
    'precipitation': 0.0,
    'wind_speed_10m': 0.0,
    'codigo_localizacao': 0.0,
    'nome_localizacao': 0.0,
    'estado': 0.0,
    'latitude': 0.0,
    'longitude': 0.0,
    'temperatura_media_localizacao': 0.0,
    'precipitacao_acumulada_localizacao': 0.0,
    'vel


**Conclusão:** a geração do relatório de execução e auditoria foi concluída com sucesso. O arquivo `quality_report.json` foi criado no diretório `output/`, contendo a data e hora da execução, a quantidade de registros e colunas processadas e os resultados das verificações de qualidade. A leitura posterior do arquivo confirmou a integridade da estrutura JSON.



## 22. Persistência dos dados em formato Parquet

**Objetivo:** armazenar os dados integrados, transformados e enriquecidos em formato Parquet, preparando a camada final do pipeline.

**Estratégia:** utilizar Pandas e PyArrow para converter o DataFrame final em arquivo Parquet, preservando os tipos de dados e permitindo armazenamento colunar eficiente.

**Justificativa:** o formato Parquet oferece compressão, armazenamento colunar e compatibilidade com ferramentas utilizadas em Engenharia de Dados.

**Resultado esperado:** gerar um arquivo Parquet na pasta `output/`, contendo os dados meteorológicos integrados e enriquecidos.


In [26]:
# Verifica a disponibilidade do PyArrow.
import pyarrow

print("PyArrow:", pyarrow.__version__)
print("Instalação validada com sucesso.")

PyArrow: 26.0.1
Instalação validada com sucesso.



**Teste:** persistência e leitura dos dados em Parquet.

O teste executa o pipeline de extração, integra as fontes, enriquece os dados com as métricas e persiste o resultado em Parquet.

Em seguida, realiza a leitura do arquivo gerado para verificar a integridade dos registros, das colunas e dos tipos de dados.

**Resultado esperado:** gerar e ler um arquivo Parquet contendo os 120 registros e as 13 colunas do conjunto de dados enriquecido.


In [27]:

# Importa as funções necessárias.
import pandas as pd

from src.extraction.pipeline import run_pipeline
from src.transformation.transform import enrich_with_metrics
from src.quality.storage import save_parquet

# Define o diretório de saída do projeto.
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

# Executa a extração dos dados.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_origem)

# Persiste o DataFrame em formato Parquet.
parquet_path = save_parquet(
    df=df_enriquecido,
    output_dir=OUTPUT_DIR,
)

# Verifica se o arquivo foi criado.
assert parquet_path.exists(), "Arquivo Parquet não foi gerado."

# Lê o arquivo Parquet para validar a persistência.
df_parquet = pd.read_parquet(
    parquet_path,
    engine="pyarrow",
)

# Verifica a quantidade de registros e colunas.
assert len(df_parquet) == len(df_enriquecido)
assert list(df_parquet.columns) == list(df_enriquecido.columns)

# Apresenta os resultados do teste.
print("Arquivo Parquet gerado:", parquet_path)
print("Tamanho:", parquet_path.stat().st_size, "bytes")
print("Registros gravados:", len(df_parquet))
print("Colunas gravadas:", len(df_parquet.columns))

# Apresenta os tipos de dados recuperados.
display(
    pd.DataFrame({
        "coluna": df_parquet.dtypes.index,
        "tipo": df_parquet.dtypes.astype(str).values,
    })
)

# Exibe uma amostra dos dados recuperados.
display(df_parquet.head())

print("\nTeste de persistência Parquet concluído.")


Arquivo Parquet gerado: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\dados_integrados.parquet
Tamanho: 10452 bytes
Registros gravados: 120
Colunas gravadas: 13


,coluna,tipo
0,time,str
1,temperature_2m,float64
2,relative_humidity_2m,int64
3,precipitation,float64
4,wind_speed_10m,float64
5,codigo_localizacao,str
6,nome_localizacao,str
7,estado,str
8,latitude,float64
9,longitude,float64


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,codigo_localizacao,nome_localizacao,estado,latitude,longitude,temperatura_media_localizacao,precipitacao_acumulada_localizacao,velocidade_media_vento_localizacao
0,2026-11-19T00:00,11.4,84,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,15.675,0.0,5.995833
1,2026-11-19T01:00,11.5,81,0.0,5.2,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,15.675,0.0,5.995833
2,2026-11-19T02:00,11.4,81,0.0,0.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,15.675,0.0,5.995833
3,2026-11-19T03:00,12.0,81,0.0,2.7,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,15.675,0.0,5.995833
4,2026-11-19T04:00,10.6,82,0.0,1.4,330110092432102,19S16W35DD1,Arkansas,33.019583,-92.722658,15.675,0.0,5.995833



Teste de persistência Parquet concluído.



**Correção:** padronização dos tipos antes da persistência.

A extração original retorna a coluna `time` como texto. Para preservar o tipo temporal no arquivo Parquet, a função `standardize_data_types()` será executada antes do enriquecimento e da gravação.

**Resultado esperado:** persistir a coluna `time` como datetime64[ns] e manter os 120 registros e as 13 colunas.


In [28]:

# Importa as funções necessárias para preparar e persistir os dados.
import pandas as pd

from src.extraction.pipeline import run_pipeline
from src.transformation.transform import (
    standardize_data_types,
    enrich_with_metrics,
)
from src.quality.storage import save_parquet

# Define o diretório principal do projeto.
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

# Executa a extração dos dados.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Padroniza os tipos de dados, incluindo a coluna temporal.
df_padronizado = standardize_data_types(df_origem)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_padronizado)

# Persiste os dados em formato Parquet.
parquet_path = save_parquet(
    df=df_enriquecido,
    output_dir=OUTPUT_DIR,
)

# Lê o arquivo Parquet para validar a persistência.
df_parquet = pd.read_parquet(
    parquet_path,
    engine="pyarrow",
)

# Verifica a integridade dos registros e das colunas.
assert len(df_parquet) == 120
assert len(df_parquet.columns) == 13

# Verifica se a coluna temporal foi preservada.
assert pd.api.types.is_datetime64_any_dtype(
    df_parquet["time"]
), "A coluna time não foi persistida como datetime."

# Verifica se as demais colunas foram preservadas.
assert list(df_parquet.columns) == list(df_enriquecido.columns)

# Apresenta os resultados da validação.
print("Arquivo Parquet:", parquet_path)
print("Tamanho:", parquet_path.stat().st_size, "bytes")
print("Registros:", len(df_parquet))
print("Colunas:", len(df_parquet.columns))
print("Tipo da coluna time:", df_parquet["time"].dtype)

# Apresenta os tipos de dados recuperados.
display(
    pd.DataFrame({
        "coluna": df_parquet.dtypes.index,
        "tipo": df_parquet.dtypes.astype(str).values,
    })
)

print("\nPersistência Parquet validada.")


Arquivo Parquet: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\dados_integrados.parquet
Tamanho: 10533 bytes
Registros: 120
Colunas: 13
Tipo da coluna time: datetime64[us]


,coluna,tipo
0,time,datetime64[us]
1,temperature_2m,float64
2,relative_humidity_2m,int64
3,precipitation,float64
4,wind_speed_10m,float64
5,codigo_localizacao,string
6,nome_localizacao,string
7,estado,string
8,latitude,float64
9,longitude,float64



Persistência Parquet validada.



**Conclusão:** a persistência dos dados em formato Parquet foi realizada com sucesso utilizando Pandas e PyArrow. O arquivo gerado contém 120 registros e 13 colunas. A leitura posterior confirmou a integridade dos dados e a preservação dos tipos, incluindo a coluna temporal `time`, armazenada como datetime64[us].



## 23. Compressão Snappy

**Objetivo:** aplicar compressão Snappy aos arquivos Parquet gerados pelo pipeline, buscando reduzir o espaço de armazenamento sem comprometer a integridade dos dados.

**Estratégia:** configurar o mecanismo de persistência PyArrow para utilizar a compressão Snappy durante a gravação dos arquivos Parquet.

**Justificativa:** a compressão permite otimizar o armazenamento dos dados e manter a compatibilidade com ferramentas utilizadas no processamento de dados.

**Resultado esperado:** gerar um arquivo Parquet comprimido com Snappy, preservando os registros, as colunas e os tipos de dados originais.



**Teste:** persistência Parquet com compressão Snappy.

O teste executa a extração, padronização e enriquecimento dos dados. Em seguida, grava o DataFrame em Parquet utilizando compressão Snappy.

A validação verifica a integridade dos registros, a preservação dos tipos de dados e o método de compressão utilizado no arquivo.


In [29]:

# Importa as bibliotecas necessárias.
import pandas as pd
import pyarrow.parquet as pq

from src.extraction.pipeline import run_pipeline
from src.transformation.transform import (
    standardize_data_types,
    enrich_with_metrics,
)
from src.quality.storage import save_parquet

# Define os diretórios do projeto.
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

# Executa a extração dos dados.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Padroniza os tipos de dados.
df_padronizado = standardize_data_types(df_origem)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_padronizado)

# Persiste os dados utilizando compressão Snappy.
parquet_path = save_parquet(
    df=df_enriquecido,
    output_dir=OUTPUT_DIR,
)

# Lê o arquivo Parquet para verificar sua integridade.
df_parquet = pd.read_parquet(
    parquet_path,
    engine="pyarrow",
)

# Obtém os metadados do arquivo Parquet.
parquet_metadata = pq.read_metadata(parquet_path)

# Identifica os métodos de compressão das colunas.
compression_methods = {
    parquet_metadata.row_group(0).column(i).compression
    for i in range(parquet_metadata.num_columns)
}

# Verifica a quantidade de registros e colunas.
assert len(df_parquet) == len(df_enriquecido)
assert len(df_parquet.columns) == len(df_enriquecido.columns)

# Verifica a preservação do tipo temporal.
assert pd.api.types.is_datetime64_any_dtype(df_parquet["time"])

# Verifica se todas as colunas utilizam Snappy.
assert compression_methods == {"SNAPPY"}

# Apresenta os resultados.
print("Arquivo:", parquet_path)
print("Tamanho:", parquet_path.stat().st_size, "bytes")
print("Registros:", len(df_parquet))
print("Colunas:", len(df_parquet.columns))
print("Tipo da coluna time:", df_parquet["time"].dtype)
print("Compressão:", compression_methods)

print("\nTeste de compressão Snappy concluído.")


Arquivo: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\dados_integrados.parquet
Tamanho: 10533 bytes
Registros: 120
Colunas: 13
Tipo da coluna time: datetime64[us]
Compressão: {'SNAPPY'}

Teste de compressão Snappy concluído.



**Conclusão:** a compressão Snappy foi aplicada com sucesso na persistência dos dados em formato Parquet, utilizando PyArrow. A inspeção dos metadados confirmou a utilização da compressão em todas as colunas. A leitura posterior preservou os 120 registros, as 13 colunas e os tipos de dados, incluindo a coluna temporal.



## 24. Particionamento temporal por ano e mês

**Objetivo:** organizar os arquivos Parquet em partições temporais, utilizando o ano e o mês da coluna `time`.

**Estratégia:** utilizar Pandas e PyArrow para extrair o ano e o mês dos registros e gerar arquivos Parquet em diretórios hierárquicos, mantendo a compressão Snappy.

**Justificativa:** o particionamento temporal facilita a organização dos dados e permite consultar períodos específicos sem precisar carregar todo o conjunto de dados.

**Resultado esperado:** gerar arquivos Parquet particionados por ano e mês, preservando os registros, as colunas e os tipos de dados.



**Teste:** particionamento temporal dos dados em Parquet.

O teste executa o pipeline de extração, padroniza os tipos de dados e aplica o enriquecimento das métricas.

Em seguida, persiste os dados particionados por ano e mês, utilizando compressão Snappy.

Serão verificadas:

- A criação da estrutura de diretórios.
- A identificação das partições de ano e mês.
- A quantidade de registros persistidos.
- A preservação das colunas e dos tipos de dados.
- A utilização da compressão Snappy.


In [30]:

# Importa as bibliotecas necessárias.
import pandas as pd
import pyarrow.parquet as pq

from src.extraction.pipeline import run_pipeline
from src.transformation.transform import (
    standardize_data_types,
    enrich_with_metrics,
)
from src.quality.storage import save_partitioned_parquet

# Define os diretórios do projeto.
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

# Executa a extração dos dados.
df_origem = run_pipeline(limit=1000, max_locations=5)

# Padroniza os tipos de dados.
df_padronizado = standardize_data_types(df_origem)

# Enriquece os dados com as métricas por localização.
df_enriquecido = enrich_with_metrics(df_padronizado)

# Persiste os dados particionados por ano e mês.
partition_dir = save_partitioned_parquet(
    df=df_enriquecido,
    output_dir=OUTPUT_DIR,
)

# Localiza todos os arquivos Parquet gerados.
parquet_files = list(partition_dir.glob("ano=*/mes=*/*.parquet"))

# Verifica se foram gerados arquivos.
assert parquet_files, "Nenhum arquivo Parquet foi encontrado."

# Lê o conjunto particionado para validar os registros.
df_particionado = pd.read_parquet(
    partition_dir,
    engine="pyarrow",
)

# Verifica a quantidade de registros.
assert len(df_particionado) == len(df_enriquecido)

# Verifica a preservação das colunas originais e das partições.
assert set(df_enriquecido.columns).issubset(
    df_particionado.columns
)
assert {"ano", "mes"}.issubset(df_particionado.columns)

# Verifica a preservação do tipo temporal.
assert pd.api.types.is_datetime64_any_dtype(
    df_particionado["time"]
)

# Inspeciona os metadados de cada arquivo Parquet.
compression_methods = set()

for parquet_file in parquet_files:
    metadata = pq.read_metadata(parquet_file)

    for row_group in range(metadata.num_row_groups):
        for column in range(metadata.num_columns):
            compression_methods.add(
                metadata.row_group(row_group)
                .column(column)
                .compression
            )

# Confirma a utilização da compressão Snappy.
assert compression_methods == {"SNAPPY"}

# Apresenta os resultados do teste.
print("Diretório:", partition_dir)
print("Arquivos Parquet:", len(parquet_files))
print("Registros persistidos:", len(df_particionado))
print("Colunas recuperadas:", len(df_particionado.columns))
print("Anos encontrados:", sorted(df_particionado["ano"].unique()))
print("Meses encontrados:", sorted(df_particionado["mes"].unique()))
print("Tipo da coluna time:", df_particionado["time"].dtype)
print("Compressão:", compression_methods)

# Exibe a estrutura das partições.
for parquet_file in parquet_files:
    print(parquet_file.relative_to(partition_dir))

print("\nTeste de particionamento concluído.")


FileExistsError: O diretório já existe: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\output\dados_particionados. Remova ou renomeie o diretório antes de executar novamente.

**Conclusão:** o particionamento temporal dos dados foi realizado com sucesso utilizando Pandas e PyArrow. Os registros foram organizados em diretórios hierárquicos por ano e mês, com compressão Snappy. A leitura do conjunto particionado confirmou a preservação dos 120 registros, das 13 colunas originais e dos tipos de dados. As colunas `ano` e `mes` foram disponibilizadas como atributos adicionais da estrutura particionada.


## 25. Orquestração do pipeline completo

**Objetivo:** integrar as etapas de extração, transformação, validação, auditoria e persistência em um único fluxo de execução.

**Estratégia:** atualizar o arquivo `run.py` para coordenar os módulos desenvolvidos em `src/`, respeitando a sequência de processamento dos dados.

**Sequência de execução:**

1. **Extração:** consumir as APIs USGS e Open-Meteo.
2. **Padronização:** ajustar os tipos de dados e formatos das colunas.
3. **Tratamento de valores nulos:** remover registros sem informações obrigatórias.
4. **Deduplicação:** identificar e remover registros duplicados.
5. **Integração:** relacionar as fontes utilizando SQL JOIN no DuckDB.
6. **Enriquecimento:** calcular métricas meteorológicas por localização.
7. **Validação:** verificar estrutura, completude, unicidade e validade dos dados.
8. **Auditoria:** gerar o relatório de execução e qualidade em formato JSON.
9. **Persistência:** armazenar os dados em Parquet, utilizando compressão Snappy e particionamento por ano e mês.

**Justificativa:** centralizar a execução do projeto, reduzir operações manuais e garantir uma sequência padronizada de processamento.

**Resultado esperado:** executar todas as etapas do pipeline com o comando `python run.py`, gerando os dados finais particionados em Parquet e o relatório de auditoria em JSON.



**Teste:** validação da execução completa do pipeline.

O teste verifica os arquivos gerados pelo `run.py`, avaliando a integridade dos dados persistidos em Parquet e a consistência do relatório de auditoria em JSON.

**Verificações previstas:**

1. Existência dos arquivos de saída.
2. Quantidade de registros e colunas do conjunto final.
3. Integridade dos tipos de dados.
4. Identificação das partições temporais.
5. Utilização da compressão Snappy.
6. Consistência das informações registradas no relatório JSON.

**Resultado esperado:** confirmar que a execução completa produziu os arquivos finais corretamente, preservando os dados e os critérios de qualidade estabelecidos.



**Teste:** integridade dos arquivos finais do pipeline.

O teste realiza a leitura dos arquivos Parquet particionados e do relatório JSON gerados pelo `run.py`.

Serão verificadas a quantidade de registros, a estrutura das colunas, os tipos de dados, a compressão Snappy, as partições temporais e os resultados da auditoria de qualidade.

**Resultado esperado:** confirmar que os arquivos finais representam corretamente os 120 registros processados pelo pipeline, preservando a estrutura e os critérios de qualidade estabelecidos.


In [ ]:

# Importa as bibliotecas necessárias para a validação.
import json
import pandas as pd
import pyarrow.parquet as pq

from pathlib import Path

# Define os diretórios e arquivos gerados pelo pipeline.
PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "output"

PARTITION_DIR = OUTPUT_DIR / "dados_particionados"
REPORT_FILE = OUTPUT_DIR / "quality_report.json"

# Verifica a existência dos arquivos finais.
assert PARTITION_DIR.exists(), "Diretório Parquet não encontrado."
assert REPORT_FILE.exists(), "Relatório JSON não encontrado."

# Localiza todos os arquivos Parquet das partições.
parquet_files = list(
    PARTITION_DIR.glob("ano=*/mes=*/*.parquet")
)

assert parquet_files, "Nenhum arquivo Parquet encontrado."

# Lê o conjunto de dados particionado.
df_final = pd.read_parquet(
    PARTITION_DIR,
    engine="pyarrow",
)

# Verifica a quantidade de registros.
assert len(df_final) == 120, "Quantidade de registros incorreta."

# Verifica as colunas originais e as colunas de particionamento.
expected_columns = {
    "time",
    "codigo_localizacao",
    "nome_localizacao",
    "estado",
    "latitude",
    "longitude",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "temperatura_media_localizacao",
    "precipitacao_acumulada_localizacao",
    "velocidade_media_vento_localizacao",
    "ano",
    "mes",
}

assert expected_columns.issubset(
    set(df_final.columns)
), "Existem colunas esperadas ausentes."

# Verifica o tipo temporal.
assert pd.api.types.is_datetime64_any_dtype(
    df_final["time"]
), "Tipo temporal incorreto."

# Verifica a quantidade de localizações.
assert df_final["codigo_localizacao"].nunique() == 5

# Inspeciona a compressão dos arquivos Parquet.
compression_methods = set()

for parquet_file in parquet_files:
    metadata = pq.read_metadata(parquet_file)

    for row_group in range(metadata.num_row_groups):
        for column in range(metadata.num_columns):
            compression_methods.add(
                metadata.row_group(row_group)
                .column(column)
                .compression
            )

assert compression_methods == {"SNAPPY"}

# Confere os valores das partições temporais.
assert df_final["ano"].nunique() >= 1
assert df_final["mes"].nunique() >= 1

# Abre o relatório de auditoria JSON.
with REPORT_FILE.open("r", encoding="utf-8") as file:
    report_data = json.load(file)

quality = report_data["data_quality"]

# Confere a consistência entre o relatório e os dados persistidos.
assert report_data["execution"]["total_records"] == len(df_final)
assert quality["total_records"] == len(df_final)
assert quality["total_columns"] == 13

# Verifica a completude.
assert quality["completeness"]["total_null_values"] == 0

# Verifica a unicidade.
assert quality["uniqueness"]["duplicate_records"] == 0

# Verifica a estrutura.
assert quality["structure"]["status"] == "válido"
assert quality["structure"]["missing_columns"] == []

# Verifica os intervalos numéricos.
assert all(
    item["invalid_count"] == 0
    for item in quality["validity"].values()
)

# Apresenta o resumo da validação.
print("VALIDAÇÃO DOS ARQUIVOS FINAIS")
print("Registros:", len(df_final))
print("Colunas:", len(df_final.columns))
print("Localizações:", df_final["codigo_localizacao"].nunique())
print("Arquivos Parquet:", len(parquet_files))
print("Compressão:", compression_methods)
print("Tipo temporal:", df_final["time"].dtype)
print("Partições:", sorted(
    df_final[["ano", "mes"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
))
print("Valores nulos:", quality["completeness"]["total_null_values"])
print("Duplicidades:", quality["uniqueness"]["duplicate_records"])

print("\nTodos os testes de integridade foram aprovados.")

VALIDAÇÃO DOS ARQUIVOS FINAIS
Registros: 120
Colunas: 15
Localizações: 5
Arquivos Parquet: 1
Compressão: {'SNAPPY'}
Tipo temporal: datetime64[us]
Partições: [(2026, 11)]
Valores nulos: 0
Duplicidades: 0

Todos os testes de integridade foram aprovados.



**Conclusão:** a execução completa do pipeline foi validada com sucesso. Os arquivos finais apresentaram 120 registros, cinco localizações e 13 colunas originais, acrescidas das duas colunas de particionamento temporal. A compressão Snappy foi confirmada nos metadados Parquet, e a leitura dos arquivos preservou os tipos de dados. O relatório JSON apresentou resultados consistentes com os dados persistidos, sem valores nulos, duplicidades ou inconsistências nos intervalos avaliados.
